In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np

file_path = '/content/drive/MyDrive/Colab Notebooks/OIBSIPDataAnalytics-L1-EDARetailSales/Employee_Dataset (1).csv'
df = pd.read_csv(file_path)

# --- 1. Produce a Data Quality Report ---
print("=== DATA QUALITY REPORT ===")
print("\n--- Count of Nulls Per Column ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows (Entire Row) ---")
print(f"Total exact duplicate rows: {df.duplicated().sum()}")

print("\n--- Duplicate Profiles (Semantic Overlap) ---")
semantic_dups = df.duplicated(subset=['First_Name', 'Last_Name', 'Join_Date', 'Email'], keep='first').sum()
print(f"Duplicate profiles based on identity fields: {semantic_dups}")

print("\n--- Data Type Profile ---")
print(df.dtypes)

print("\n--- Value Range Anomalies (Numeric Columns) ---")
print(df[['Age', 'Salary', 'Phone']].describe())

# Save pre-cleaning baseline metrics for summary table
before_metrics = {
    'row_count': len(df),
    'null_count': df.isnull().sum().sum(),
    'dup_count': df.duplicated(subset=['First_Name', 'Last_Name', 'Join_Date', 'Email'], keep='first').sum(),
    'dtypes': df.dtypes.to_dict()
}

# --- 2. Data Type Correction & Standardisation (Pre-Imputation Setup) ---
df['Join_Date'] = pd.to_datetime(df['Join_Date'])
df['Employee_ID'] = df['Employee_ID'].astype(str)
df['Phone'] = df['Phone'].abs().astype(str)  # Fix negative formatting artifact

# Break down compound feature into standard relational schema (1NF)
df[['Department', 'Region']] = df['Department_Region'].str.split('-', expand=True)
df.drop(columns=['Department_Region'], inplace=True)

# --- 3. Missing Data Handling (Imputation Strategy) ---
# Impute numeric fields conditionally based on relevant grouping factor medians
df['Salary'] = df.groupby('Department')['Salary'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df.groupby('Performance_Score')['Age'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df['Age'].astype(int)

# --- 4. Duplicate Removal ---
df.drop_duplicates(subset=['First_Name', 'Last_Name', 'Join_Date', 'Email'], keep='first', inplace=True)

# --- 5. Outlier Detection (IQR Method) ---
print("\n=== OUTLIER DETECTION (IQR METHOD) ===")
for col in ['Age', 'Salary']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}: Found {len(outliers)} outliers outside range ({lower_bound:.2f} to {upper_bound:.2f}) -> Retained valid variations.")

# --- 6. Produce a Before vs. After Summary Table ---
after_metrics = {
    'row_count': len(df),
    'null_count': df.isnull().sum().sum(),
    'dup_count': df.duplicated(subset=['First_Name', 'Last_Name', 'Join_Date', 'Email'], keep='first').sum(),
    'dtypes': df.dtypes.to_dict()
}

summary_df = pd.DataFrame({
    'Metric': ['Row Count', 'Total Nulls', 'Duplicate Profiles'],
    'Before Cleaning': [before_metrics['row_count'], before_metrics['null_count'], before_metrics['dup_count']],
    'After Cleaning': [after_metrics['row_count'], after_metrics['null_count'], after_metrics['dup_count']]
})

print("\n=== BEFORE VS. AFTER METRICS SUMMARY ===")
print(summary_df.to_string(index=False))

print("\n=== DATA TYPE CONVERSION ANALYSIS ===")
for col in before_metrics['dtypes']:
    if col in df.columns:
        print(f"Column: {col.ljust(18)} | Before: {str(before_metrics['dtypes'][col]).ljust(10)} | After: {str(df[col].dtype)}")

# --- 7. Save Cleaned Dataset ---
output_path = '/content/drive/MyDrive/Colab Notebooks/OIBSIPDataAnalytics-L1-EDARetailSales/Cleaned_Employee_Dataset.csv'
df.to_csv(output_path, index=False)
print(f"\nExecution complete. Cleaned dataset saved directly to Drive path: '{output_path}'.")

=== DATA QUALITY REPORT ===

--- Count of Nulls Per Column ---
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

--- Duplicate Rows (Entire Row) ---
Total exact duplicate rows: 0

--- Duplicate Profiles (Semantic Overlap) ---
Duplicate profiles based on identity fields: 6

--- Data Type Profile ---
Employee_ID           object
First_Name            object
Last_Name             object
Age                  float64
Department_Region     object
Status                object
Join_Date             object
Salary               float64
Email                 object
Phone                  int64
Performance_Score     object
Remote_Work             bool
dtype: object

--- Value Range Anomalies (Numeric Columns) ---
              Age        